In [1]:
!pwd

/home/jovyan/work


In [2]:
!ls

'1. Introduccion.ipynb'			    Untitled.ipynb
'2. DataFrame.ipynb'			    emulador_datos.py
'2. ProcesamientoParalelomnn_final.ipynb'   spark-warehouse
'3. Practica Mysql Hdfs Spark.ipynb'


In [3]:
!python emulador_datos.py

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("LambdaBatch")
    .getOrCreate()
)

mysql_url = "jdbc:mysql://mysql:3306/retail_db"

properties = {
    "user": "root",
    "password": "root",
    "driver": "com.mysql.cj.jdbc.Driver"
}

# =====================================================
# 1. INGESTA MYSQL -> RAW
# =====================================================

orders = (
    spark.read
    .jdbc(
        url=mysql_url,
        table="orders",
        properties=properties
    )
)

order_items = (
    spark.read
    .jdbc(
        url=mysql_url,
        table="order_items",
        properties=properties
    )
)

orders.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/lambda/raw/orders"
)

order_items.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/lambda/raw/order_items"
)

In [ ]:
# =====================================================
# 2. RAW -> CLEANSED
# =====================================================

orders_raw = (
    spark.read
    .parquet("hdfs://namenode:9000/lambda/raw/orders")
)

items_raw = (
    spark.read
    .parquet("hdfs://namenode:9000/lambda/raw/order_items")
)

In [ ]:
from pyspark.sql.functions import col, trim

orders_cleansed = (
    orders_raw
    .filter(col("order_id").isNotNull())
    .withColumn("status", trim(col("status")))
)

items_cleansed = (
    items_raw
    .filter(col("order_item_order_id").isNotNull())
    .filter(col("quantity") > 0)
)

In [ ]:
orders_cleansed.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/lambda/cleansed/orders"
)

items_cleansed.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/lambda/cleansed/order_items"
)

In [ ]:
# =====================================================
# 3. JOIN
# =====================================================

orders_clean = (
    spark.read
    .parquet("hdfs://namenode:9000/lambda/cleansed/orders")
)

items_clean = (
    spark.read
    .parquet("hdfs://namenode:9000/lambda/cleansed/order_items")
)

ventas_batch = (
    orders_clean
    .join(
        items_clean,
        orders_clean.order_id ==
        items_clean.order_item_order_id,
        "inner"
    )
)

In [ ]:
ventas_batch.write.mode("overwrite").parquet(
    "hdfs://namenode:9000/lambda/batch/ventas"
)

In [ ]:
import time

while True:

    print("================================")
    print("INICIANDO BATCH")
    print("================================")

    # MySQL -> RAW
    orders = (
        spark.read
        .jdbc(
            url=mysql_url,
            table="orders",
            properties=properties
        )
    )

    order_items = (
        spark.read
        .jdbc(
            url=mysql_url,
            table="order_items",
            properties=properties
        )
    )

    orders.write.mode("overwrite").parquet(
        "hdfs://namenode:9000/lambda/raw/orders"
    )

    order_items.write.mode("overwrite").parquet(
        "hdfs://namenode:9000/lambda/raw/order_items"
    )

    # RAW -> CLEANSED
    orders_clean = (
        orders
        .filter(col("order_id").isNotNull())
    )

    items_clean = (
        order_items
        .filter(col("order_item_order_id").isNotNull())
    )

    orders_clean.write.mode("overwrite").parquet(
        "hdfs://namenode:9000/lambda/cleansed/orders"
    )

    items_clean.write.mode("overwrite").parquet(
        "hdfs://namenode:9000/lambda/cleansed/order_items"
    )

    # JOIN
    ventas = (
        orders_clean
        .join(
            items_clean,
            orders_clean.order_id ==
            items_clean.order_item_order_id,
            "inner"
        )
    )

    ventas.write.mode("overwrite").parquet(
        "hdfs://namenode:9000/lambda/batch/ventas"
    )

    print("BATCH FINALIZADO")
    print("Esperando 10 minutos...")

    time.sleep(600)